# The Combine Exchange Rate: reproducibility notebook

This notebook rebuilds every number, table and figure in our writeup, *The Combine Exchange Rate: how many NFL games is a Combine worth?*, from the raw competition files. Every analysis script runs again and writes into a fresh folder; besides the raw data, the only inputs are frozen files such as the preregistrations. The notebook then checks the rebuilt numbers in two ways:

1. **The number checker** (`writeup/check_numbers.py`, Section 8) recomputes each number quoted in the writeup from the fresh results, at the rounding we quote. It fails on any mismatch, and on any number in the text that no check covers.
2. **A full comparison** (Section 10) checks every fresh results file against the copy committed in our repository, value by value.

**How the notebook is laid out.** Each section explains one step in plain language and runs it with one line of code. A step runs one of the repository's own scripts in a separate Python process, with pinned package versions and every statistical setting as committed: the same permutation, bootstrap, simulation and cross-validation counts, and the same seeds. The plumbing (environment, data discovery, logs, comparison) is in `notebooks/run_all.py`, printed in Appendix D.

**Run time and stages.** The full chain took 4.3 to 5.7 hours in each of our four local runs on a 14-core laptop (Section 11). At our central estimate of Kaggle's speed that is about 18 hours of compute, more than one 12-hour Kaggle CPU session. On Kaggle, the same notebook therefore runs as four stages:

| Stage | Steps | Needs |
|-|-|-|
| A | data cache, game frames, round-1 probes, E007, E010, E011, speed calibration | the competition data |
| B | E008, E009 (transfer map) | stage A's saved output |
| C | E012 (Combine Game Equivalents) | stage A's saved output |
| D | exports, figures, number checker, comparison | the saved outputs of stages B and C |

Each stage saves its working folder as notebook output for the next stage. The version attached to the writeup runs stage D with the outputs of stages A–C attached, and every step reports where it ran and how long it took.

## 0. Setup

`STAGE` picks the part of the pipeline this session runs: `"all"` on a machine without a time limit, or one Kaggle stage (`"A"`, `"B"`, `"C"`, `"D"`). Setup takes four actions:
- it finds the code bundle (an attached Kaggle dataset, or this repository) and the competition data;
- it builds or checks the pinned interpreter: CPython 3.13.5 and the exact package versions of `notebooks/requirements-kaggle.txt`, with any mismatch stopping the run;
- it creates the working folder, merging the saved outputs of earlier stages;
- it verifies that the compatibility edits listed in Appendix A are already in the repository's scripts, and stops if one is missing.

In [ ]:
STAGE = "all"  # Kaggle: "A", "B", "C" or "D" (see the table above)

In [ ]:
import sys
import zipfile
from pathlib import Path


def locate_pipeline() -> Path:
    """notebooks/ of the code bundle: beside this notebook locally, an attached dataset on Kaggle (tree or zip)."""
    for c in (Path.cwd(), Path.cwd() / "notebooks"):
        if (c / "run_all.py").is_file() and (c.parent / "src" / "bdb27").is_dir():
            return c
    found = sorted(Path("/kaggle/input").rglob("notebooks/run_all.py"))
    if not found:
        for z in sorted(Path("/kaggle/input").rglob("bdb27-bundle*.zip")):
            zipfile.ZipFile(z).extractall("/tmp/bdb27-bundle")
        found = sorted(Path("/tmp/bdb27-bundle").rglob("notebooks/run_all.py"))
    if not found:
        raise FileNotFoundError("attach the bdb27 code bundle dataset (notebooks/README.md)")
    return found[0].parent


sys.path.insert(0, str(locate_pipeline()))
import run_all as ra  # noqa: E402
from IPython.display import Markdown, display  # noqa: E402

ctx = ra.setup(STAGE)

## 1. Raw data to a Parquet cache (S01, S02)

**S01** converts the nine competition CSVs to Parquet once, with explicit column types. It stops if any row count differs from the data release we analysed.

**S02** builds the game frames most game-side measures start from:
- every regular-season frame after the snap of every tracked player;
- direction standardized (offense moving toward +x);
- smoothed speed and tangential acceleration added.

In [ ]:
for step in ("S01", "S02"):
    ra.run(ctx, step)

## 2. Round 1: the sanity link and five Combine-to-game probes (S03, S04a–S04e)

**E001** is the pipeline's sanity link: tracked 40 peak speed against game top speed. It also writes the shared feature store.

**E002–E006** are the five preregistered first-round probes:
- WR/TE route breaks (E002);
- DL first-step burst (E003);
- OL lateral reversals (E004);
- DB backpedal transitions (E005);
- the shape of the 40's sprint profile (E006).

They build every Combine signature and game trait that the transfer map retests (writeup Section 4; Section 4 here). Their split-half reliabilities of the game measures feed the writeup's "13 of the 16 pass the reliability gate".

These five scripts were written for the first version of our test harness. Its result keys were renamed before round 2, so the notebook renames them in its working copies (Appendix A). No feature, target or statistical setting changes. Their round-1 p-values came from that first harness and cannot be reproduced by the current one. The writeup uses none of them; Section 10 lists them as documented differences.

In [ ]:
for step in ("S03", "S04a", "S04b", "S04c", "S04d", "S04e"):
    ra.run(ctx, step)

## 3. Hidden top speed, drill intensity and Combine ball tracking (S05–S07)

- **E007** times each receiver's tracked 40 from the official 20-yard line to the finish, then ranks receivers by speed beyond their 40 time. The hook's "23.2 mph" and "the fastest finish for his 40 time of 91 Combine receivers" come from its leaderboard.
- **E010** measures how close to his own 40 peak each player runs his position drills. It is stable, yet predicts neither play speed nor snaps (writeup Section 5).
- **E011** is the first look at the 2025 Combine ball-tracking rows (writeup Section 6, league item 3).

In [ ]:
for step in ("S05", "S06", "S07"):
    ra.run(ctx, step)

**Calibration: how long will the next stages take on this machine?** Kaggle does not publish its CPU, and stage C (E012) is the longest single session. Stage A therefore ends by timing one fixed calculation: E001's receiver link at full settings (50 × 5 cross-validation, 2,000 and 10,000 permutations, three workers, as every script runs). The step checks the result against ours, then scales the time by our local runs to predict each stage's hours on this machine. If stage C is predicted above 10.5 hours, do not start it: E012 would first have to be split across sessions (`notebooks/README.md`, Run times).

In [ ]:
if ctx.stage in ("all", "A") and ra.done(ctx, "S01", "S03"):
    ra.calibrate(ctx)

## 4. The transfer map (S08, S09)

**E008** re-estimates all 18 Combine-to-game links under the current protocol. The links are the 14 round-1 primaries plus the sanity link at four positions. For each link it computes:
- one-sided Freedman–Lane permutation tests beyond the stopwatch and the draft slot;
- the reliability of both measures, and reliability-corrected bounds;
- the drill reliability profile behind Figure 5.

**E009** repeats the 18 tests beyond the stopwatch alone and measures how much the draft slot moves each partial correlation.

Together they give writeup Section 4, most of Section 5, and Figures 4–5.

In [ ]:
for step in ("S08", "S09"):
    ra.run(ctx, step)

## 5. Combine Game Equivalents (S10)

**E012** is the analysis the writeup is built on. For each of 12 game traits it does four things:
- it splits per-game values into a stable player level and game-to-game noise;
- it prices four priors in games, with cross-validation and a 1,000-resample player bootstrap: the position average; plus stopwatch and measurements; plus draft slot; plus Combine tracking;
- it runs the pseudo-prospective test on 2025 rookies, fit on 2023–24;
- it simulates each trait's design to calibrate the intervals (post hoc).

E012 produces Table 1, Figures 2–3, writeup Sections 2–3, the Receiver A case study, and the "worth N games" numbers in Sections 6–7. It is the longest step, so on Kaggle it has a stage of its own (stage C).

In [ ]:
ra.run(ctx, "S10")

## 6. Two exports for the writeup (S11a, S11b)

Two small post hoc exports read stored outputs only:
- Receiver A's forecasts under each prior, and the pools behind the hook's "11 of 15", from E012's stored 2025 forecasts;
- a receiver-cluster bootstrap of E007's same-40 receiver pairs, which cross-checks the hook's rank.

In [ ]:
for step in ("S11a", "S11b"):
    ra.run(ctx, step)

## 7. Figures (S12)

`writeup/make_figures.py` draws Figures 1–5 from the fresh results alone; no statistic is recomputed. It asserts that every number a figure prints matches the writeup's rounding. On Kaggle the font falls back to DejaVu Sans, so the images differ slightly from ours, pixel by pixel, but not in content.

In [ ]:
ra.run(ctx, "S12")
if ra.done(ctx, "S12"):
    ra.show_figures(ctx)

## 8. Every quoted number, checked (S13)

`writeup/check_numbers.py` runs on the fresh results. It does four things:
- it recomputes each number quoted in the writeup from its source key, at the rounding we quote;
- it asserts that the exact phrase carrying the number is in the text;
- it fails on any number that no check covers;
- it enforces the word and figure budgets.

Exit code 0 means every number in the writeup was reproduced. Appendix B has the full output, one line per number with its fresh source value.

In [ ]:
ra.run(ctx, "S13")
if ra.done(ctx, "S13"):
    ra.show_checker(ctx)

**Table 1** as published. The number checker above asserts every cell against the fresh E012 results.

In [ ]:
if ra.done(ctx, "S13"):
    ra.show_table1(ctx)

## 9. Trait definitions and position averages (S14)

The writeup's Code line promises these. The definitions are the preregistered ones (`experiments/results/E012_prereg.json`). Each position average is the mean, over E012's analysis players, of each player's average over his qualifying games; the three class means follow it. The step asserts that its player counts equal E012's.

Several traits are centered within route, alignment or coverage, or are measured over expected, so their averages sit near zero by construction. These averages are not quoted in the writeup.

In [ ]:
ra.run(ctx, "S14")
if ra.done(ctx, "S14"):
    ra.show_traits(ctx)

## 10. Comparison with the committed results

Every results file regenerated above is compared with the version committed in our repository, value by value:
- **Must be identical:** integers, strings, booleans and structure.
- **Floating-point values** must agree to a relative 1e-9. That allows only last-digit differences from summation order (BLAS and OpenMP threads, CPU type).
- **Documented differences** are allowed only where a cause is recorded, and each one is listed with its cause. For example, the round-1 p-values come from the first version of the harness and cannot be reproduced by the current one.
- **Anything else** fails the file. A file that passes only because of a documented defect (a known reproducibility bug in a value no writeup number uses) is marked PASS*.

In [ ]:
if ra.done(ctx, *[s.id for s in ra.STEPS]):
    report = ra.compare(ctx)
else:
    print("The comparison runs once every step is done (stage D on Kaggle).")

One quoted number rests on values that cannot be regenerated. "About 90 first-round tests (q 0.06)" uses the smallest round-1 p-values of the other first-round tests, which E008 holds as constants copied from the round-1 results. Those p-values came from the first harness, which no longer exists. The cell below checks each constant against the committed round-1 file it was copied from.

In [ ]:
if ra.done(ctx, "S08"):
    ra.transcribed_constants(ctx)

## 11. Where and how long each step ran

CPU time includes the parallel workers. Peak memory is the largest process.

In [ ]:
display(Markdown(ra.timing_table(ctx)))
ra.finish(ctx)

---
## Appendix A. Compatibility edits

These edits are committed in the repository's scripts; setup only verifies they are present, so `patches.diff` below stays empty. None of them touches a feature, target, sample or statistical setting:
- **The round-1 scripts (E002–E006).** They were written for the first harness's result keys, which were renamed before round 2. E005's figure also used a helper that was later removed. The replacement computes the same OLS residuals.
- **E009.** It checks that its rerun reproduces E008's leave-one-class-out results. The check was written against the committed E008.json, which predates a field the current harness adds. On a freshly regenerated E008.json the check failed on that extra field, so it now compares the same two fields on both sides.
- **E011.** It read its frozen plan from a temporary file. The same plan is committed inside `E011.json["plan"]` and as `experiments/results/E011_prereg.json`, which E011 now reads (setup checks the two are equal).
- **E012.** It checks its implementation choices against a freeze file. That file is now committed under `experiments/results/E012_audit/`, with the same SHA-256.
- **The pairs export.** It used an absolute path on one machine.

In [ ]:
print((ctx.run_dir / "patches.diff").read_text())

## Appendix B. Number checker output (S13)

In [ ]:
if ra.done(ctx, "S13"):
    ra.show_log(ctx, "S13")

## Appendix C. Environment

These are the interpreter and package versions every step ran with, each checked against `notebooks/requirements-kaggle.txt`.

In [ ]:
import json

for p in sorted(ctx.run_dir.glob("env_*.json")):
    env = json.loads(p.read_text())
    print(f"{p.name}: {env['platform']}, {env['cpu_count']} CPUs, workers {env['workers']}\n{env['report']}")

## Appendix D. The pipeline code

This is `notebooks/run_all.py`, which the cells above call. The analysis itself lives in the repository's `src/bdb27/` package and its `experiments/` and `writeup/` scripts, which are in the attached code bundle.

In [ ]:
print(Path(ra.__file__).read_text())